# 12 · Context Compression

**Idea:** even relevant passages carry filler. **Compression** trims retrieved text to the sentences that actually matter before the generator sees it — cutting tokens (cost) and easing the *lost-in-the-middle* effect. We use extractive compression: keep the sentences most similar to the query.

> Runs on Ollama. Shows token savings and an end-to-end answer via `rag_answer`.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

In [ ]:
from rag_lab import load_markdown, NumpyStore, format_context, compress_context
from study_buddy.tokens import count_tokens

corpus = load_markdown()
store = NumpyStore(provider='ollama'); store.add(corpus)

query = 'what is the lost-in-the-middle effect and how is it mitigated?'
hits = store.search(query, k=4)

Compare the raw retrieved context against the compressed version (2 best sentences per hit) — and count the tokens saved.

In [ ]:
raw = format_context(hits)
small = compress_context(query, hits, keep=2, provider='ollama')

print('raw tokens:       ', count_tokens(raw))
print('compressed tokens:', count_tokens(small))
print('\n--- compressed context ---\n')
print(small)

Now answer the question from the **compressed** context with the shared `rag_answer` pipeline. Fewer tokens in the prompt, same answer.

In [ ]:
from rag_lab import rag_answer
from study_buddy import CostTracker

tracker = CostTracker()
result = rag_answer(query, store, provider='ollama', k=4, context=small, tracker=tracker)
print(result.answer)
print('\ncost:', tracker.summary())

Optionally answer with a hosted model (DeepSeek) to feel the token savings as real money. Skips if no key.

In [ ]:
try:
    cloud = CostTracker()
    r = rag_answer(query, store, provider='deepseek', k=4, context=small, tracker=cloud)
    print(r.answer)
    print('\ncost:', cloud.summary())
except Exception as e:
    print('deepseek skipped:', type(e).__name__, e)

**Takeaway:** compression is a cheap win — fewer tokens means lower cost and less distraction, usually with no loss in answer quality. `rag_answer` is the retrieve→generate pipeline the frontier notebooks build on.

Next → `13_parent_child_chunking.ipynb`